In [0]:
%run ../../utils/00_silver_utils

In [0]:
# Widget params; run_id comes from bronze via get_shared_run_id so this notebook's audit rows share bronze's run_id.
dbutils.widgets.text("silver_config_path", "/Volumes/cdr_silver/config/silver_config/silver_config.json")
dbutils.widgets.text("table_filter", "")

silver_config_path = dbutils.widgets.get("silver_config_path")
table_filter_raw = dbutils.widgets.get("table_filter").strip()
table_filter = {t.strip().lower() for t in table_filter_raw.split(",") if t.strip()} if table_filter_raw else None

dbutils.widgets.text("run_id", "")
run_id_param = dbutils.widgets.get("run_id").strip()
run_id = run_id_param if run_id_param else str(uuid.uuid4())
if not run_id_param:
    logger.warning(f"No run_id widget value supplied -- generating standalone run_id={run_id}")

logger.info(f"run_id              = {run_id}")
logger.info(f"silver_config_path  = {silver_config_path}")
logger.info(f"table_filter        = {table_filter or 'ALL TABLES FOR THIS NOTEBOOK'}")

In [0]:
# Loads silver_config.json and filters it down to the tables tagged silver_notebook == "clinical_events".
with open(silver_config_path, "r") as f:
    SILVER_CONFIG = json.load(f)

LAYER = "clinical_events"
tables_for_this_notebook = [
    t for t, cfg in SILVER_CONFIG.items() if cfg["silver_notebook"] == LAYER
]
if table_filter:
    tables_for_this_notebook = [t for t in tables_for_this_notebook if t in table_filter]

logger.info(f"Processing {len(tables_for_this_notebook)} table(s) for layer '{LAYER}': {tables_for_this_notebook}")

ensure_silver_audit_table(spark)

In [0]:
#  transform for omr (doesn't fit the generic rule engine): groups Height/Weight/BMI, drops missing/extreme values, drops existing BMI rows, recomputes BMI from height+weight per subject/chartdate/seq_num.
from pyspark.sql.functions import when as _when, lower as _lower

def clean_omr(df):
    df = df.withColumn(
        "result_category",
        _when(_lower(col("result_name")).contains("height"), lit("Height"))
        .when(_lower(col("result_name")).contains("weight"), lit("Weight"))
        .when(_lower(col("result_name")).contains("bmi"), lit("BMI"))
        .otherwise(col("result_name"))
    )

    df = df.filter(col("result_value").isNotNull() & (trim(col("result_value")) != ""))

    # Cast result_value to a proper numeric column now, before any further use --
    # everything below (range filter, BMI calc) reads this numeric column instead of
    # re-casting the original string column over and over.
    df = df.withColumn("result_value", expr("try_cast(result_value as double)"))

    df = df.filter(
        ~(
            ((col("result_category") == "Height") & ((col("result_value") < 50) | (col("result_value") > 250))) |
            ((col("result_category") == "Weight") & ((col("result_value") < 2) | (col("result_value") > 300)))
        )
    )

    df_no_bmi = df.filter(col("result_category") != "BMI")

    heights = (df_no_bmi.filter(col("result_category") == "Height")
               .select("subject_id", "chartdate", "seq_num", col("result_value").alias("height_cm")))
    weights = (df_no_bmi.filter(col("result_category") == "Weight")
               .select("subject_id", "chartdate", "seq_num", col("result_value").alias("weight_kg")))

    bmi = (heights.join(weights, ["subject_id", "chartdate", "seq_num"])
           .withColumn("result_value", col("weight_kg") / ((col("height_cm") / 100) ** 2))
           .withColumn("result_category", lit("BMI"))
           .withColumn("result_name", lit("BMI (recomputed)"))
           .select("subject_id", "chartdate", "seq_num", "result_name", "result_category", "result_value"))

    return df_no_bmi.unionByName(bmi, allowMissingColumns=True)

In [0]:
# Maps each table needing bespoke logic (not expressible as a generic cleaning_rules entry) to its function.
SPECIAL_TRANSFORMS = {
    "omr": clean_omr
}

In [0]:
# Builds cdr_silver.mimic_iv.dim_interpretation -- same catalog/schema as every other silver table (confirmed).
# microbiologyevents.interpretation itself is left RAW/unchanged in silver; this table is for downstream joins
# (assumption -- flagged, since the guide didn't explicitly say whether to overwrite the raw column too).
# MIMIC-IV documents S/R/I/P; bronze also has D/N/Z/NULL which aren't documented -- all four map to 'Unknown'.
def build_dim_interpretation(spark, run_id):
    rows = [
        ("S", "Susceptible"), ("R", "Resistant"), ("I", "Intermediate"), ("P", "Pending"),
        ("D", "Unknown"), ("N", "Unknown"), ("Z", "Unknown"), ("NULL", "Unknown"),
    ]
    dim_df = spark.createDataFrame(rows, ["interpretation_code", "interpretation_standardized"])
    start_time = datetime.now()
    try:
        result = write_silver(spark, dim_df, "dim_interpretation", ["interpretation_code"], run_id)
        end_time = datetime.now()
        logger.info(f"SUCCESS | dim_interpretation -> {result['target_table']} | {result['load_type']} | {result['row_count']} rows")
        return Row(
            run_id=run_id, layer=LAYER, source_table="dim_interpretation", target_table=result["target_table"],
            load_type=result["load_type"], status="SUCCESS", row_count=result["row_count"],
            rows_inserted=result["rows_inserted"], rows_updated=result["rows_updated"],
            start_time=start_time, end_time=end_time,
            duration_seconds=(end_time - start_time).total_seconds(), error_message=None,
        )
    except Exception as e:
        end_time = datetime.now()
        logger.error(f"FAILED | dim_interpretation | {e}")
        return Row(
            run_id=run_id, layer=LAYER, source_table="dim_interpretation", target_table="dim_interpretation",
            load_type=None, status="FAILED", row_count=None, rows_inserted=None, rows_updated=None,
            start_time=start_time, end_time=end_time,
            duration_seconds=(end_time - start_time).total_seconds(), error_message=str(e)[:2000],
        )

pre_loop_audit_rows = [build_dim_interpretation(spark, run_id)]

In [0]:
# Runs every table for this layer: read bronze, clean, write/merge to silver, collect an audit row, then write the audit log.
audit_rows = list(pre_loop_audit_rows)
success_count = 0
failure_count = 0

for table_name in tables_for_this_notebook:
    audit_row = process_table(
        spark, table_name, SILVER_CONFIG, run_id, layer=LAYER,
        special_transform=SPECIAL_TRANSFORMS.get(table_name),
    )
    audit_rows.append(audit_row)
    if audit_row.status == "SUCCESS":
        success_count += 1
    else:
        failure_count += 1

write_audit_rows(spark, audit_rows)

logger.info(
    f"Layer '{LAYER}' complete | run_id={run_id} | {success_count} succeeded | "
    f"{failure_count} failed | {len(tables_for_this_notebook)} total tables"
)
if failure_count > 0:
    logger.warning(f"{failure_count} table(s) failed -- check {SILVER_AUDIT_TABLE} (run_id = {run_id})")